# 共用机器人运动学：base.py
阅读顺序：相邻位姿 → N系位姿 → 相邻角速度 → N系速度。
$R_{PC}$ 把子系 C 的分量变换到父系 P；$T_{PC}$ 的平移列也在父系中表示。
$N$ 原点位于名义车轴高度平面，故 $p_C^N=[x,y,0]^T$，地面高度为 $-r$。
$No,Co,Bo,Lo,Ro$ 分别是 N,C,B,WL,WR 的原点。左轮沿 $+C.y$、右轮沿 $-C.y$。
$l$ 是质心距车轴的高度；累计前向行程统一记为 $s$。
本文件不做小角度近似。修改 `base.py` 后统一重建所有笔记本。

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'base.py').exists():
    if root==root.parent: raise RuntimeError('找不到 theory/base.py')
    root=root.parent
sys.path.insert(0,str(root))
import sympy as s
import numpy as np
from IPython.display import display, Math
# 只改 LaTeX 显示，不替换计算符号：dx -> x 上点，ddx -> x 双点。
from base import q, qd, qdd
latex_names = {}
for coordinate, velocity, acceleration in zip(q, qd, qdd):
    name = s.latex(coordinate)
    latex_names[velocity] = r'\dot{' + name + '}'
    latex_names[acceleration] = r'\ddot{' + name + '}'
s.init_printing(symbol_names=latex_names)
np.set_printoptions(precision=8,suppress=True,linewidth=130)
def show(label, expression):
    display(Math(label+' = '+s.latex(expression, symbol_names=latex_names)))


In [2]:
from base import *
k=kinematics()
show("q",q)
show(r"\dot q",qd)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 相邻坐标系位姿和角速度矩阵
角速度矩阵 $[\omega_{child/parent}^{parent}]_\times=\dot R R^T$，各项仍是精确三角函数。

In [3]:
for name,T in k['adjacent'].items():
    show('T_{'+name+'}',T)
    show(r'[\omega_{'+name+r'}]_{\times}',k['relative_omega_hat'][name])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 各刚体在 N 系的位置、速度和角速度
$$T_{NB}=T_{NC}T_{CB},\quad T_{NWL}=T_{NC}T_{CWL},\quad T_{NWR}=T_{NC}T_{CWR}.$$
$$v_i^N=\frac{\partial p_i^N}{\partial q}\dot q,
\quad[\omega_{i/N}^N]_\times=\dot R_{Ni}R_{Ni}^T.$$
下方是代码实际输出的展开矩阵；两个动力学场景均读取这些结果。

In [4]:
for name in ('C','B','WL','WR'):
    show('T_{N'+name+'}',k['T'][name])
    show('v_{'+name+'}^{N}',k['v'][name])
    show(r'\omega_{'+name+'}^{N}',k['omega'][name])
    show(r'[\omega_{'+name+r'}^{N}]_{\times}',k['omega_hat'][name])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 接触点速度与三个独立约束
接触点指当前接触位置的轮上材料点，速度用刚体公式计算，不能把持续变化的几何接触位置直接对时间求导。

In [5]:
expressions,Acon,contact=constraints()
for name,value in contact.items():show('v_{contact,'+name+'}^{N}',value)
show('A_c\dot q',expressions)
show('A_c',Acon)

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_20719/1265846320.py:3: SyntaxWarning: invalid escape sequence '\d'
  show('A_c\dot q',expressions)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 能量接口
$$T_i=\tfrac12m_i(v_i^N)^Tv_i^N+\tfrac12(\omega_i^N)^T R_{Ni}I_iR_{Ni}^T\omega_i^N.$$
这里只显示逐刚体精确能量，近似方式由两个场景分别决定。

In [6]:
parts,T,V=energies()
for name,value in parts.items():show('T_{'+name+'}',value)
show('V',V)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>